(set OpenAi client and add API key from the colab internal variables)

In [177]:
!pip install openai

import openai
from google.colab import userdata

api_key = userdata.get('OPENAI_API_KEY')

client = openai.OpenAI(api_key=api_key)

## 1. Upload the PDF
(a functionality that allows us to upload a PDF file)

In [178]:
from google.colab import files
import io
uploaded = files.upload()

Saving drf-reac-gems.pdf to drf-reac-gems.pdf


## 2. Extract the text from the PDF
(functionality that extracts the text from the PDF file as raw text)

In [179]:
!pip install pdfminer.six

In [180]:
from pdfminer.high_level import extract_text

pdf_file_name = list(uploaded.keys())[0]
raw_text = extract_text(pdf_file_name)

print(raw_text[:500])


DRF React Gems - E-commerce Platform Documentation

Project Overview

DRF React Gems is a full-stack e-commerce platform built with Django REST Framework

backend and React frontend. The platform implements user authentication, shopping cart

functionality, wishlist management, payment processing, order history tracking, and

asynchronous email notifications using Celery and Redis.

Technology Stack

Backend: Django, Django REST Framework

Frontend: React

Database: PostgreSQL

Authentication: J


## 3. Chunk (punctual) the text
(I need a function for punctual chunking that will be used for RAG)

In [181]:
import re

def punctual_chunking(text, chunk_size=500, overlap=100):
    split_points = r'(?<=[.!?;\n])\s*'
    sentences = re.split(split_points, text)

    chunks = []
    current_chunk = ""
    i = 0
    while i < len(sentences):
        sentence = sentences[i]

        if len(current_chunk) + len(sentence) < chunk_size:
            current_chunk += sentence
            i += 1
        else:
            if current_chunk:
                chunks.append(current_chunk.strip())
                overlap_start = max(0, len(current_chunk) - overlap)
                current_chunk = current_chunk[overlap_start:] + sentence
                i += 1
            else:
                 chunks.append(sentence[:chunk_size].strip())
                 sentences[i] = sentence[chunk_size:]
                 if not sentences[i]:
                     i += 1

    if current_chunk:
        chunks.append(current_chunk.strip())

    return chunks

punctual_chunks = punctual_chunking(raw_text, chunk_size=600, overlap=100)
print(f"Created {len(punctual_chunks)} chunks using punctual chunking.")
print("First chunk:", punctual_chunks[0])

Created 24 chunks using punctual chunking.
First chunk: DRF React Gems - E-commerce Platform Documentation
Project Overview
DRF React Gems is a full-stack e-commerce platform built with Django REST Framework
backend and React frontend.The platform implements user authentication, shopping cart
functionality, wishlist management, payment processing, order history tracking, and
asynchronous email notifications using Celery and Redis.Technology Stack
Backend: Django, Django REST Framework
Frontend: React
Database: PostgreSQL
Authentication: JWT (JSON Web Tokens)
Cloud Services: Azure (hosting), Redis Cloud (caching)
Media Storage: Cloudinary


## 4. Generate embeddings
(I need a function that uses OpenAI embedding model ada 002 to turn our punctual chunks into embeddings and return object which will be chunk - embeddings array.)

In [182]:
import openai
import time

def generate_embeddings(chunks, client, model="text-embedding-ada-002"):
    chunk_embeddings = {}
    try:
        response = client.embeddings.create(
            input=chunks,
            model=model
        )
        for i, chunk in enumerate(chunks):
            if i < len(response.data):
                chunk_embeddings[chunk] = response.data[i].embedding
            else:
                print(f"Warning: No embedding returned for chunk {i+1}.")

        return chunk_embeddings
    except Exception as e:
        print(f"An error occurred during embedding generation: {e}")
        return None

embeddings = generate_embeddings(punctual_chunks, client)

if embeddings:
   print(f"Generated embeddings for {len(embeddings)} chunks.")
   first_chunk = list(embeddings.keys())[0]
   print("\nFirst Chunk:")
   print(first_chunk)
   print("\nEmbedding for the first chunk (first 10 elements):")
   print(embeddings[first_chunk][:10])

Generated embeddings for 24 chunks.

First Chunk:
DRF React Gems - E-commerce Platform Documentation
Project Overview
DRF React Gems is a full-stack e-commerce platform built with Django REST Framework
backend and React frontend.The platform implements user authentication, shopping cart
functionality, wishlist management, payment processing, order history tracking, and
asynchronous email notifications using Celery and Redis.Technology Stack
Backend: Django, Django REST Framework
Frontend: React
Database: PostgreSQL
Authentication: JWT (JSON Web Tokens)
Cloud Services: Azure (hosting), Redis Cloud (caching)
Media Storage: Cloudinary

Embedding for the first chunk (first 10 elements):
[0.006997174583375454, -0.021773481741547585, -0.023849714547395706, -0.04095841199159622, -0.005729863420128822, 0.0045805200934410095, -0.009423940442502499, -0.032680444419384, -0.019225377589464188, -0.009659876115620136]


## 5. Store embeddings into Vector Database
(Set up the Chroma DB by installing, importing and creating a client. Also create a collection with a name 'my_document_embeddings'.)

### 1. Set up Chroma DB

In [191]:
!pip install chromadb

import chromadb

client_db = chromadb.Client()

collection = client_db.get_or_create_collection(name="drf_react_gems")

print(f"Chroma DB client created and collection '{collection.name}' is ready.")

Chroma DB client created and collection 'drf_react_gems' is ready.


### 2. Store the embeddings and chunks
(we should store in chroma in our collection the pairs that we created earlier)

In [192]:
ids = [f"chunk_{i}" for i in range(len(embeddings))]
documents = list(embeddings.keys())
embedding_vectors = list(embeddings.values())

collection.add(
    embeddings=embedding_vectors,
    documents=documents,
    ids=ids
)

print(f"Added {len(documents)} documents and embeddings to the collection.")
print(f"Collection count: {collection.count()}")

Added 24 documents and embeddings to the collection.
Collection count: 24


## 6. Vector search functionality
(I need a function that accepts a text and converts it into embeddings. Then returns them. This will be used later for the vector search in the DB.)

### 1. Convert the query into embeddings

In [193]:
import openai

def generate_query_embedding(query_text, client, model="text-embedding-ada-002"):
    try:
        response = client.embeddings.create(
            input=[query_text],
            model=model
        )
        if response.data and len(response.data) > 0:
            return response.data[0].embedding
        else:
            print("Warning: No embedding returned for the query.")
            return None
    except Exception as e:
        print(f"An error occurred during query embedding generation: {e}")
        return None

query = "What is DRF React Gems?"
query_embedding = generate_query_embedding(query, client)

if query_embedding:
   print(f"Generated embedding for the query (first 10 elements):")
   print(query_embedding[:10])

Generated embedding for the query (first 10 elements):
[0.00045814947225153446, -0.013483901508152485, -0.03019705042243004, -0.028345614671707153, -0.010721101425588131, 0.00621809484437108, -0.0013275794917717576, -0.022977888584136963, -0.01573002152144909, -0.0019823990296572447]


### 2. Vector Search
(I need another function that does a vector search in the Chroma. It does the vector search using the result of the function generate_query_embedding. I also need to control the tollerence of the search.)

In [194]:
def vector_search(query_embedding, collection, n_results=5, distance_threshold=None):
    if query_embedding is None:
        print("Error: Query embedding is None.")
        return None, None

    try:
        where_clause = {}
        if distance_threshold is not None:
             where_clause = {"distance": {"$lt": distance_threshold}}

        results = collection.query(
            query_embeddings=[query_embedding],
            n_results=n_results,
            include=['documents', 'distances'],
            # where=where_clause
        )

        if results and results.get('documents') and results['documents'][0]:
            return results['documents'][0], results['distances'][0]
        else:
            print("No results found for the query.")
            return [], []

    except Exception as e:
        print(f"An error occurred during vector search: {e}")
        return None, None

query_text = "What is DRF React Gems?"
query_embedding = generate_query_embedding(query_text, client)

if query_embedding:
    search_results, distances = vector_search(query_embedding, collection, n_results=3, distance_threshold=0.2)

    if search_results:
        print(f"\nTop {len(search_results)} most relevant chunks for the query '{query_text}':")
        for i, chunk in enumerate(search_results):
            print(f"Result {i+1} (Distance: {distances[i]:.4f}):\n{chunk}\n---")


Top 3 most relevant chunks for the query 'What is DRF React Gems?':
Result 1 (Distance: 0.2452):
DRF React Gems - E-commerce Platform Documentation
Project Overview
DRF React Gems is a full-stack e-commerce platform built with Django REST Framework
backend and React frontend.The platform implements user authentication, shopping cart
functionality, wishlist management, payment processing, order history tracking, and
asynchronous email notifications using Celery and Redis.Technology Stack
Backend: Django, Django REST Framework
Frontend: React
Database: PostgreSQL
Authentication: JWT (JSON Web Tokens)
Cloud Services: Azure (hosting), Redis Cloud (caching)
Media Storage: Cloudinary
---
Result 2 (Distance: 0.3786):
(JSON Web Tokens)
Cloud Services: Azure (hosting), Redis Cloud (caching)
Media Storage: Cloudinary
Frontend Hosting: Firebase
Error Monitoring: Sentry
Email Service: Gmail SMTP
Background Tasks: Celery with Redis
Live URLs
Main Application: https://drf-react-gems.web.app
Admin P

### 3. AI response
(I need a new function thata ccepts the result from the vector search and uses OpenAI 4o-mini model to)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [251]:
history = []

def generate_ai_response(query_text, search_results, client, model="gpt-4o-mini"):
    if not search_results:
        return "Could not find relevant information to answer the query."

    context = "\n\n".join(search_results)

    try:
        response = client.responses.create(
            model=model,
            temperature=0.1,
            input=[
                {"role": "system", "content": "You are a helpful assistant expert at answering questions strictly from provided context, which consists of retrieved documents relevant to the query. Use only the information in the context. If the query's answer is not in the context, respond exactly with: 'I do not know.' It is very important to keep responses under 70 words. It is also very important, to always end with a complete sentence without cutting off mid-thought, mid-paragraph, or mid-sentence."},
                {"role": "user", "content": f"Based on the following context, answer the query:\n\nContext:\n{context}\n\nQuery: {query_text}"}
            ],
            max_output_tokens=100,
            top_p=0.1,
            stream=True,
        )
        # return response.output_text

        collected_output = ""
        for event in response:
            if event.type == "response.output_text.delta":
                # print each chunk as it arrives (no newline, flush immediately)
                print(event.delta, end="", flush=True)
                collected_output += event.delta
            elif event.type == "response.completed":
                break

        print()  # newline after streaming finishes
        return collected_output

    except Exception as e:
        print(f"An error occurred during AI response generation: {e}")
        return None

# query = "What is DRF React Gems?"
# query_embedding = generate_query_embedding(query, client)

# if query_embedding:
#     search_results, distances = vector_search(query_embedding, collection, n_results=3)

# if search_results:
#     ai_response = generate_ai_response(query, search_results, client)
#     if ai_response:
#         print("\nAI Response:")
#         print(ai_response)
# else:
#     print("No search results to generate a response.")

## 7. Ask qustions
(Use all the functions one after another as it should first accept a qustion and then process it and print the response)

In [252]:
query = input("Please enter your question about the document: ")
print(f"\nUser Query: {query}")

query_embedding = generate_query_embedding(query, client)

if query_embedding:
    print(f"\nQuery Embedding (first 10 elements): {query_embedding[:10]}...")

    search_results, distances = vector_search(query_embedding, collection, n_results=5, distance_threshold=0.4)

    if search_results:
        print(f"\nVector Search Results ({len(search_results)} chunks found):")
        print(f"Result 1 (Distance: {distances[0]:.4f}):\n{search_results[0][:200]}...")
        print(f"Result 2 (Distance: {distances[1]:.4f}):\n{search_results[1][:200]}...")

        ai_response = generate_ai_response(query, search_results, client)

        if not ai_response:
            print("\nFailed to generate response.")
            # print(ai_response)
        else:
            print("Failed to generate AI response.")
    else:
        print("No relevant information found in the document.")
else:
    print("Failed to generate query embedding.")

Please enter your question about the document: my name is bea

User Query: my name is bea

Query Embedding (first 10 elements): [-0.03162612020969391, -0.013481480069458485, 0.006213204003870487, -0.030974840745329857, -0.013781068846583366, 0.02665034681558609, -0.023406976833939552, 0.0066560739651322365, -0.013976452872157097, 0.0001643462455831468]...

Vector Search Results (5 chunks found):
Result 1 (Distance: 0.5663):
ge report
is available at https://beatrisilieva.github.io/drf-react-gems/coverage-report/index.html.To run tests: coverage run manage.py test && coverage report
Deployment Architecture
Deployment Serv...
Result 2 (Distance: 0.5710):
s and React components
The frontend contains 14 page components
Database Design
Database Technology
The system uses PostgreSQL as the primary database for all application data storage.Data Normalizati...
An error occurred during AI response generation: Responses.create() got an unexpected keyword argument 'conversation'

Failed to genera